#SFbox in python#

It would be interesting to be able to do scf modeling of interfacial phenomena using the python environmnet. This jupyter notebook comes together with a file called sf_scf.py. In this document there is a class called sfbox. This class is imported by: 'from sf_scf import sfbox'. Of course the sf_scf.py document should be in the same directory as the directory form where this notebook is loaded.

The idea is that sfbox works like a black box. The language that is used to specify a problem for sfbox is defined in these classes (see the __init__ statements) so the user can not make mistakes. Currently (as an example) the classes of MyInput are the following. 

*****

class MyInput:    #sfbox definitions: do not touch

    class lattice:
        def __init__(lat,name):
            lat.name=name; lat.n_layers=[]; lat.gradients=1;
            lat.geometry="planar"; lat.bc_lb=[]; lat.bc_ub=[];
              
    class segment:
        def __init__(seg,name,size):
            seg.name=name; seg.size=size; seg.valence = 0; seg.epsilon=80
            seg.freedom = "free"; seg.pinned_range =""; seg.frozen_range=""
        
    class molecule:
        def __init__(mol,name,composition):
            mol.name = name; mol.composition = composition; 
            mol.freedom=""; mol. phibulk=-1; mol.theta=-1;
    
    class output:
        def __init__(out,name):
            out.name=name; out.outlist=[]; out.append=True;
            
    class system:
        def __init__(sys,name,segs):
            sys.name=name; sys.segs=segs; 
            sys.chi=np.zeros ((len(segs),len(segs))); sys.charged=False;
        def set_chi(sys,A,B,value):
            sys.chi[A][B]=sys.chi[B][A]=value
**** 

The idea is that these classes show the user exactly the 'name' of relevant 'properties.
The second part of the following cell the following happens:

- it examplifies how these classes are used to define the problem (polymer brush).
- generate a reference to the class sfbox
- start the calculations in a series wherein the chainlength is varied.

When appropriate the user can analys results by mining (any computed and stored quantity) from sfbox. (an example is shown below).

note that the 'machinery' of sfbox now only fills the phi vector of the molecules with random numbers. Hence, there are no calculations done yet (work in progress), but the content can easily be carried over from PySFbox.py (I hope). 

Food for thought.... Enjoy.

In [ ]:
import numpy as np
from sf_scf import sfbox

class MyInput: #sfbox definitions: do not touch
    class lattice:
        def __init__(lat,name):
            lat.name=name; lat.n_layers=[]; lat.gradients=1;
            lat.geometry="planar"; lat.bc_lb=[]; lat.bc_ub=[];
              
    class segment:
        def __init__(seg,name,size):
            seg.name=name; seg.size=size; seg.valence = 0; seg.epsilon=80
            seg.freedom = "free"; seg.pinned_range =""; seg.frozen_range=""
        
    class molecule:
        def __init__(mol,name,composition):
            mol.name = name; mol.composition = composition; 
            mol.freedom=""; mol. phibulk=-1; mol.theta=-1;
    
    class output:
        def __init__(out,name):
            out.name=name; out.outlist=[]; out.append=True;
            
    class system:
        def __init__(sys,name,segs):
            sys.name=name; sys.segs=segs; 
            sys.chi=np.zeros ((len(segs),len(segs))); sys.charged=False;
        def set_chi(sys,A,B,value):
            sys.chi[A][B]=sys.chi[B][A]=value
     
 #Specify your input starting here.   
    lats = []
    lats.extend([lattice("bol")]) 
    lats[-1].geometry="spherical"
    lats[-1].n_layers.append(100); lats[-1].bc_lb.append("mirror"); lats[-1].bc_ub.append("mirror")

    segs = []
    segs.extend([segment("A",2)]); A=0
    segs.extend([segment("W",1)]); W=1
    segs.extend([segment("X",2)]); X=2   
    segs[-1].freedom = "pinned"
    segs[-1].pinned_range : "[1;1]" #perhaps better in python language
    
    mols = []
    mols.extend([molecule("water","(w)1")]); water=0;
    mols[-1].freedom="solvent"
    mols.extend([molecule("pol","(X)1(A)99")]); pol=1;
    mols[-1].freedom="restricted"; mols[-1].theta=5;
    
    outs=[] #optional if you want output files in addition to 'data mining'
    outs.extend([output("kal")]) 
    outs[-1].outlist.append( "mol : * : phibulk")
    outs[-1].outlist.append( "sys : NN : grand potential")
    
    outs.extend([output("pro")])
    outs[-1].append=False
    outs[-1].outlist.append( "mol : * : phi")
    outs[-1].outlist.append( "sys : NN : grand potential density")
    
    syss=[]
    syss.extend([system("NN",segs)])
    syss[-1].set_chi(A,W,0.5)
    syss[-1].set_chi(X,W,0.4)
# done with input definitions:

    
I=MyInput()       #call your input
sf=sfbox(MyInput) #call sfbox

Nrange=np.linspace(10,30,3).astype(np.int_)
for N in Nrange:
    I.mols[I.pol].composition="(X)1(A)"+str(N-1)
    print("working on:", I.mols[I.pol].composition)
    sf.doit("silent") #choose 'verbose' or 'silent'



In [ ]:
import matplotlib.pyplot as plt
m=sf.lats[-1].n_layers[0]
z=np.linspace(1,m,m)
plt.plot(z,sf.mols[I.pol].phi)
plt.xlabel('z')
plt.ylabel('$\phi$')
plt.show()

